In [0]:
import json
import time
import pyspark.sql.functions as sf
from functools import reduce

CHECKPOINT_PATH = "/Volumes/dev/raw/checkpoints/azure-trip-updates_checkpoint_microbatch.json"
DATA_PATH = "/Volumes/dev/raw/ttc_trip_updates_volume"

In [0]:
def get_sorted_files(path: list[str]) -> list[str]:
    """Return .jsonl files sorted oldest -> newest."""
    files = [f.name for f in dbutils.fs.ls(path) if f.name.endswith(".jsonl")]
    files.sort(key=lambda f: time.strptime(f.split("_")[2].replace(".jsonl", ""), "%Y%m%dT%H"))
    return files


def load_checkpoint() -> dict:
    try:
        with open(CHECKPOINT_PATH, "r") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {"last_file": None, "last_timestamp": 0}


def save_checkpoint(last_file: str, last_timestamp) -> None:
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump({"last_file": last_file, "last_timestamp": last_timestamp}, f)


def read_file_from_timestamp(file: str, from_timestamp) -> tuple:
    """Read rows newer than from_timestamp and return (df, max_timestamp)."""
    df = spark.read.json(f"{DATA_PATH}/{file}")
    df = df.filter(sf.col("data.timestamp") > from_timestamp)
    df = df.withColumn("source_name", sf.lit(file))
    df = df.withColumn("ingested_at", sf.lit(int(time.time())))
    # Extract date hour from file name, e.g., trip_updates_20260305T00
    date_hour = file.split("_")[2].replace(".jsonl", "")
    df = df.withColumn("date_hour", sf.lit(date_hour))
    max_ts = df.agg(sf.max("data.timestamp").alias("max_ts")).collect()[0]["max_ts"]
    return df, max_ts


def micro_batch() -> list:
    """
    Read all unprocessed files in chronological order.

    - Files older than the checkpointed file are skipped entirely.
    - The checkpointed file is read from last_timestamp onward.
    - All newer files are read in full (from timestamp 0).
    """
    checkpoint = load_checkpoint()
    last_file = checkpoint["last_file"]
    last_timestamp = checkpoint["last_timestamp"]

    all_files = get_sorted_files(DATA_PATH)  # oldest -> newest

    # Determine where to start processing
    if last_file is None:
        # No checkpoint: process everything from the beginning
        start_idx = 0
        start_timestamp = 0
    elif last_file in all_files:
        start_idx = all_files.index(last_file)
        start_timestamp = last_timestamp
    else:
        # Checkpointed file no longer exists; start fresh
        start_idx = 0
        start_timestamp = 0

    fetched_data = []

    for i, file in enumerate(all_files[start_idx:], start=start_idx):
        # Use the stored timestamp only for the checkpointed file itself
        from_timestamp = start_timestamp if i == start_idx else 0

        df, max_ts = read_file_from_timestamp(file, from_timestamp)

        if max_ts is not None:  # skip empty/fully-filtered files
            fetched_data.append(df)
            save_checkpoint(file, max_ts)

    return reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), fetched_data) if fetched_data else None

In [0]:
while True:
    time.sleep(30)
    df = micro_batch()
    if df:
        df.write.format("delta").mode("append").partitionBy("date_hour").saveAsTable("dev.raw.trip_updates_microbatch")
